In [1]:
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from multiprocessing import Pool
from tqdm.notebook import tqdm; tqdm.pandas()
from recs import *

DATASET_SIZE='large'

data = DataLoader(data_size=DATASET_SIZE) 
histories, imprs, labels = data.init_padded_behaviors(data.test)

Available device: GPU 



## Create sparse vectors with TF-IDF

The following functionality includes stemming, stopwords removal, including bigrams and trigrams, and sets min and max document frequencies. It returns a sparse vector representations of 20,000 dimensions.

In [2]:
# Vectorise titles using TF-IDF
def get_news_vectors_TFIDF():
    stemmer = SnowballStemmer('english', ignore_stopwords=True)
    tfidf = TfidfVectorizer(max_features=20000,
                            stop_words=stopwords.words('english'),
                            preprocessor=stemmer.stem,
                            ngram_range=(1,3),
                            max_df=0.9, min_df=5)

    news_vectors = tfidf.fit_transform(data.news_titles.numpy()) 
    return news_vectors

news_vectors = get_news_vectors_TFIDF()

In [3]:
# For one
top_n = 3
user, candidate = histories[1], imprs[1]
user_vector = np.mean(news_vectors[user], axis=0) # this mean value is the user profile 
candidate_vector = news_vectors[candidate]
similar = cosine_similarity(np.asarray(user_vector), candidate_vector.toarray())[0]
top_indices = similar.argsort()[::-1][:top_n]
ranked_news_ids = np.take(candidate, top_indices)
np.array(data.news_titles.numpy())[ranked_news_ids]

array([b'One New Thing to Check Out on Every Hawaiian Island',
       b"Stephanie Parze's Car Is at Home. So Is Her Phone. But She's Missing.",
       b'New Trump rule to make more health care rates public'],
      dtype=object)

## Calculate recommendations

This loops over the behaviours rows. It represents users as the mean of their readings histories. This representation is then compared with each item in the candidate list. This results in a similarity score which is then used for ranking. 

Due to the high dimensionality of the vectors, we cannot use vectorised implementations (matrix multiplication) for cosine similarity calculations. 

In [4]:
# 10m large / 2m30 small
def compute_recommendations(hist, imps, news_vectors):
    rec_lst = []
    for user, candidate_id in tqdm(zip(hist, imps), total=len(hist)): 
        
        # User representation
        user_vector = np.mean(news_vectors[user], axis=0)

        # News representation
        news_vector = news_vectors[candidate_id]

        # Map User x News
        similarities = cosine_similarity(np.asarray(user_vector), news_vector.toarray())[0]
        rec_lst.append(similarities)
        
    return rec_lst

rec_lst = compute_recommendations(histories, imprs, news_vectors)

  0%|          | 0/251821 [00:00<?, ?it/s]

In [5]:
len(rec_lst), rec_lst[0].shape

(251821, (25,))

## Evaluate

For the evaluation procedure, we need equal length vectors. Thus, we pad every candidate impression and label sequence to equal lengths, and then pass them to our log and evaluation function. 

In [6]:
# padding on the indices and labels
col_len = data.max_imps # padding is set in `data` object
row_len = len(labels)

true = np.zeros((row_len, col_len))
recs = np.zeros((row_len, col_len))
for i, (rec, lab) in enumerate(zip(rec_lst, labels)):
    pad_rec = list(rec)
    pad_lab = list(lab)
    if len(lab) < col_len:
        pad_lab = lab + [0.] * (col_len - len(lab))
        pad_rec = list(rec) + [0.] * (col_len - len(rec))
    true[i, :col_len] = pad_lab[:col_len]
    recs[i, :col_len] = pad_rec[:col_len]

In [7]:
log_results(labels, recs, imprs, 'CB-Filtering', dataset_size=DATASET_SIZE)

Saved model predictions here: ../.data/CB-Filtering.npy


,CB-Filtering
modelname,CB-Filtering
dataset_size,large
timestamp,2025-03-28 14:31
auc,0.5751
mean_mrr,0.2273
ndcg@5,0.2318
ndcg@10,0.2979
mean_epc,1.5386
mean_intra_list_diversity,0.2462
mean_surprisal,9.5014
